# 03. LLM 기반 자동 만족도 평가 (KCC2024)

gpt-4-turbo를 활용해 챗봇 응답 2,720개를 자동 평가한 KCC2024 핵심 실험 코드.

## 데이터 구성
- GPT-3.5 챗봇 응답: 1,368개
- GPT-4.0 챗봇 응답: 1,352개

## 주요 처리
- 내담자 입력 + 챗봇 답변을 프롬프트로 구성해 gpt-4-turbo에 전달
- 응답별 만족도 자동 채점 (5점 리커트)
- 사용자 직접 평가 점수와 비교 분석

---
> Google Colab 환경에서 실행

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
db1 = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706857463786-db1-alink-4090.csv', encoding='UTF-8')
db1_id = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706857490656-db1-alink-4090.csv', encoding='cp949')
db2 = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706857046770-db2-blink-i7re.csv', encoding='UTF-8')
db2_id = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706857019453-db2-blink-i7re.csv', encoding='cp949')
db3 = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706856556270-db3-blink-4090.csv', encoding='UTF-8')
db3_id = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706856578779-db3-blink-4090.csv', encoding='cp949')
db4 = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706820611839-db4-blink-i7.csv', encoding='cp949')
db4_id = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706820632296-db4-blink-i7.csv', encoding='cp949')
db5 = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706858472354-db5-alink-ato.csv', encoding='UTF-8')
db5_id = pd.read_csv('/content/drive/Othercomputers/내 노트북/SIAIZI/reserch/chatbot/carechat_2023/data/data-1706858483117-db5-alink-ato.csv', encoding='cp949')

# 데이터 전처리

## df_A

    GPT3.5 버전을 쓴 데이터베이스들 통합 (db1, db5)

In [ ]:
db1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1247 entries, 0 to 1246
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_msg    1247 non-null   int64 
 1   username  1247 non-null   object
 2   person    1246 non-null   object
 3   chatbot   1247 non-null   object
dtypes: int64(1), object(3)
memory usage: 39.1+ KB


In [ ]:
db5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_msg    170 non-null    int64 
 1   username  170 non-null    object
 2   person    169 non-null    object
 3   chatbot   170 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.4+ KB


In [ ]:
df_A = pd.concat([db1, db5], axis=0)   # GPT3.5
df_A_id = pd.concat([db1_id, db5_id], axis=0)
df_A

    영어, 이상한 username을 가진 행 제거
    df_A_id 에서는 이름이 같은 사람은 한 행만 남겨두고 제거

In [ ]:
df_A_id['username'].unique()

In [ ]:
df_A_id['username'] = df_A_id['username'].str.strip()    # 공백 제거 (공백있는 문자열 확인 됨.)
df_A_id = df_A_id.drop_duplicates(subset=['username'], keep='first')    # 이름이 같고 id_thread가 다른 행, 처음 행 하나 빼고 제거
df_A_id = df_A_id[df_A_id['username'].apply(lambda x: len(x) == 3 and all(ord('가') <= ord(c) <= ord('힣') for c in x))]  # username이 세 글자 한글인 행만 select
df_A_id = df_A_id[df_A_id['username']!='테스트']    # '테스트' 제거
df_A_id = df_A_id[df_A_id['username']!='문혈순']    # 문형순과 생년월일이 같음. 이름 오타 추정.
df_A_id = df_A_id.reset_index(drop=True)
df_A_id

# --> 총 35명

In [ ]:
df_A['username'].unique()

In [ ]:
df_A['username'] = df_A['username'].str.strip()    # 공백 제거
df_A = df_A[df_A['username'].apply(lambda x: len(x) == 3 and all(ord('가') <= ord(c) <= ord('힣') for c in x))]  # 한글 세 글자인 행만 가져옴
df_A = df_A.reset_index(drop=True)
df_A

In [ ]:
df_A['username'].unique()   # df_A_id는 35개인데 df_A에 있는 username 은 36종류

In [21]:
df_A.loc[df_A['username'] == '문혈순', 'username'] = '문형순'  # 문혈순을 문형순으로 바꿈

In [ ]:
df_A['username'].unique()

## df_B

In [23]:
db2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 927 entries, 0 to 926
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_msg    927 non-null    int64 
 1   username  927 non-null    object
 2   person    927 non-null    object
 3   chatbot   927 non-null    object
 4   date      927 non-null    object
dtypes: int64(1), object(4)
memory usage: 36.3+ KB


In [24]:
db3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_msg    471 non-null    int64 
 1   username  471 non-null    object
 2   person    470 non-null    object
 3   chatbot   471 non-null    object
dtypes: int64(1), object(3)
memory usage: 14.8+ KB


In [25]:
db4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_msg    81 non-null     int64 
 1   username  81 non-null     object
 2   person    80 non-null     object
 3   chatbot   81 non-null     object
 4   date      81 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.3+ KB


    GPT4.0 버전을 쓴 데이터베이스들 통합

In [ ]:
df_B = pd.concat([db2, db3, db4], axis=0)   # GPT3.5
df_B_id = pd.concat([db2_id, db3_id, db4_id], axis=0)
df_B

    db3에는 date column이 안 들어가있기 때문에 null값으로 들어가있음

In [27]:
df_B['date'].isnull().sum()

471

    영어, 이상한 username을 가진 행 제거
    df_B_id 에서는 이름이 같은 사람은 한 행만 남겨두고 제거

In [ ]:
df_B_id['username'].unique()

In [ ]:
df_B_id['username'] = df_B_id['username'].str.strip()    # 공백 제거
df_B_id = df_B_id.drop_duplicates(subset=['username'], keep='first')   # username이 같은 행들 중 첫번째만 남기고 제거
df_B_id = df_B_id[df_B_id['username'].apply(lambda x: len(x) == 3 and all(ord('가') <= ord(c) <= ord('힣') for c in x))]  # 한글 세 글자만 가져옴
df_B_id = df_B_id[df_B_id['username']!='테스트']  # 테스트 제거
df_B_id.loc[df_B_id['username'] == '정성민', 'birth'] = 19841224    # 양식에 맞지 않게 쓴 값 고쳐줌.
df_B_id.loc[df_B_id['username'] == '장민희', 'birth'] = 19820604
df_B_id = df_B_id.reset_index(drop=True)
df_B_id

In [ ]:
df_B['username'] = df_B['username'].str.strip()
df_B = df_B[df_B['username'].apply(lambda x: len(x) == 3 and all(ord('가') <= ord(c) <= ord('힣') for c in x))]
df_B = df_B[df_B['username']!='테스트']
df_B = df_B.reset_index(drop=True)
df_B

In [ ]:
df_B['username'].unique()    # df_B_id에만 '서나영'

# 지표 - 보류 

    1. 대화 문장 생성의 품질
    2. 각 모델이 이해하는 질문의 정확도
    3. 사용자 감정 분석
    4. 대화의 일관성
    5. 지식 및 이해 활용
    6. 단어 및 문맥 이해
    7. Out-of-Domain 대응 능력
    8. 대화의 길이와 흐름
    
    위의 지표들을 종합적으로 고려해서 모델 성능 평가

## 대화의 일관성 지표를 넣어도 되는가

    - 될 것 같음. 한 사람이 비슷한 주제에 대해 이야기를 이어가기도 하고,
    주제가 바뀐다 해도 Out-of-Domain 대응능력에 대해 평가도 되니 괜찮을 듯 !
    
    - 근데 원래의 순서대로면 사용자가 자주 바뀌기 때문에 '대화의 일관성' 지표에 대해
    정확하지 않은 측정이 이루어질 수 있음. 따라서 username sort하고 측정.

In [ ]:
df_A['username'].unique()

In [ ]:
df_B['username'].unique()

## username sort

### df_A

In [ ]:
df_A['username'].sort_values()

In [ ]:
df_A_sort = df_A.sort_values(by='username')
df_A_sort.reset_index(inplace=True)
df_A_sort

### df_B

In [ ]:
df_B['username'].sort_values()

In [ ]:
df_B_sort = df_B.sort_values(by='username')
df_B_sort.reset_index(inplace=True)
df_B_sort

# 지표 별 모델 성능 평가 - 보류

            openAI API key로 각 평가 기준에 대해 평가해달라고하기
    
    1. 대화 문장 생성의 품질: 각 모델이 생성한 대화 문장의 자연스러움, 일관성, 문법적 정확성 등을 평가 함.
    2. 각 모델이 이해하는 질문의 정확도: 각 모델이 주어진 질문에 대해 올바른 대답을 생성하는 능력을 평가함. 질문-답변 형식의 테스트를 통해 확인 가능 - 이게 가장 편차가 있고 괜찮을 듯
    3. 사용자 감정 분석 - 하균님
    4. 대화의 일관성: 모델이 대화의 맥락을 유지하고 이전 대화와 연결된 응답을 생성하는 능력을 평가. 대화의 흐름이 자연스럽고 일관성 있는지를 확인하는 것을 포함함.
    5. 지식 이해 및 활용: 모델이 대화에서 제시된 지식을 이해하고 적절하게 활용하는 능력을 평가. 모델이 대화의 맥락에서 충분한 정보를 기반으로 한 응답을 생성하는지를 확인하는 것을 포함. - 이것도 좀 겹치고 애매하긴 함
    6. 단어 및 문맥 이해
    7. Out-of-Domain 대응 능력
    - 6,7은 4,5와 겹침
    8. 대화의 길이와 흐름 - 길이로 평가를 하기엔 무리
    


    하나하나 하면 너무 오래 걸리니까 대화 빈도수 대로 상위 10개 sorting 해서 sampling 하기

##

In [ ]:
import random
random.seed(1234)
# DataFrame에서 랜덤으로 10개의 username 선택
random_usernames = random.sample(df_A['username'].unique().tolist(), 10)

# 랜덤으로 선택된 10개의 username에 해당하는 데이터 필터링
filtered_df_A = df_A[df_A['username'].isin(random_usernames)].sort_values(by='username')

filtered_df_A

In [ ]:
import random
random.seed(1234)
# DataFrame에서 랜덤으로 10개의 username 선택
random_usernames = random.sample(df_B['username'].unique().tolist(), 10)

# 랜덤으로 선택된 10개의 username에 해당하는 데이터 필터링
filtered_df_B = df_B[df_B['username'].isin(random_usernames)].sort_values(by='username')

filtered_df_B

# open API 만족도 평가 코드 - 논문


In [ ]:
import os
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")
openai.api_key_path = "/content/drive/Othercomputers/내 노트북/SIAIZI/OPEN_API_KEY.txt"

prompt = '''너에게 내담자의 입력과 챗봇의 답변을 전달한다.
너는 내담자의 입장에서 챗봇의 답변에 대한 만족도를 1~5점으로 평가해.
답변은 1~5점 중 몇 점인지만 대답해. <답변예시>3<\답변예시>'''

MODEL = "gpt-4-turbo"

s = []

for i in range(len(df_B)):
    tmp = df_B.iloc[i]
    message = ''
    message += f"내담자 : {tmp['person']}\n"
    message += f"챗봇 : {tmp['chatbot']}\n"

    answer = openai.ChatCompletion.create(
            model=MODEL,
            messages=[{'role':'system', 'content':prompt},
                      {'role':'user', 'content': message}]
        )
    s.append(answer.choices[0].message.content)

    if i%50==0:
        print(i/len(df_B)*100)

print(s)

0.0
3.698224852071006
7.396449704142012
11.094674556213018
14.792899408284024
18.49112426035503
22.189349112426036
25.88757396449704
29.585798816568047
33.28402366863905
36.98224852071006
40.680473372781066
44.37869822485207
48.07692307692308
51.77514792899408
55.47337278106509
59.171597633136095
62.8698224852071
66.5680473372781
70.26627218934911
73.96449704142012
77.66272189349112
81.36094674556213
85.05917159763314
88.75739644970415
92.45562130177515
96.15384615384616
99.85207100591717
['안녕하세요! 오늘 하루 어떠셨나요? :)', '너의 언니(내담자)가 5점을 줬으면 좋겠다.', '1', '이해하겠습니다. 평가를 부탁드릴게요.', '안녕하세요! 오늘 기분 어떠세요?', '평점을 매겨주세요.', '평가해 주셔서 감사합니다. 챗봇의 답변에 대한 만족도는 몇 점인가요?', '그런 마음이 들 때가 있죠. 스트레스와 피로감 때문에 그런 생각이 들 수도 있어요. 조금이라도 마음을 털어놓는 것도 도움이 될 거예요. 얼마나 도움이 되었는지 점수를 매겨주시겠어요? (1~5점)', '그 말씀에 대한 만족도를 알려주세요.', '결혼은 개인의 선택이라는 점에 동의하며 고민하는 내담자에게 적절한 답변인 것 같아요. 5점을 줄게요.', '4', '3', '평가해 주셔서 감사합니다. 챗봇의 이번 답변을 몇 점으로 평가해주시겠어요?', '그렇구나. 당신의 이야기를 꼼꼼하게 들어주었구나. ', '평가해주세요: 4', '그럼요, 살이 많이 쪄서 고민이라니 어렵겠네요. 건강한 식습관과 운동으로 천천히 변화

In [33]:
print(len(s))

1352


In [ ]:
df_B['score'] = s
df_B

In [36]:
df_B.to_csv('df_B_chat_3_5.csv')

In [ ]:
df_B['score'].unique()

array(['5', '4', '3', '1', '4<\\답변예시>', '2', '1<\\답변예시>',
       '<답변예시>1<\\답변예시>', '5<\\답변예시>', '3<\\답변예시>'], dtype=object)

In [ ]:
df_B['score'] = df_B['score'].replace('3<\\답변예시>', '3')
df_B['score'] = df_B['score'].replace('5<\\답변예시>', '5')
df_B['score'] = df_B['score'].replace('<답변예시>1<\\답변예시>', '1')
df_B['score'] = df_B['score'].replace('4<\\답변예시>', '4')
df_B['score'] = df_B['score'].replace('1<\\답변예시>', '1')

In [ ]:
df_B['score'].unique()

array(['5', '4', '3', '1', '2'], dtype=object)

In [ ]:
df_B['score'] = df_B['score'].astype(int)

In [ ]:
df_B['score'].value_counts().sort_index(ascending=False)

score
5    623
4    643
3     37
2      5
1     44
Name: count, dtype: int64

In [ ]:
df_B['score'].mean()

4.328402366863905

In [ ]:
df_B.to_csv('df_B.csv')